# COMP3516 Spring Group Project

**Notebook note:** this version keeps the original workflow while making a
small number of modelling and evaluation changes:

1. **Session-cluster grouped stratified split** - the OctoNet data clusters
   into ~70 short "sessions" (same user, same activity, recordings within 60s
   of each other). A random or ordered split mixes closely related samples
   between train and val, which makes validation optimistic. Cell 4 now clusters first and
   splits whole sessions.
2. **Fixed 50-epoch training** — the grouped validation split is still used
   to report accuracy, but each training run goes the full 50 epochs.
3. **Bidirectional LSTMs** — the recurrent encoders read each sequence in
   both directions while keeping the same hidden size, layer count, and
   dropout.

Comments below explain the purpose of each non-trivial step.


# COMP3516 2026 Spring Group Project
## Multi-modal human activity recognition

In this task, we will utilize the OctoNet dataset developed by HKU AIoT Lab to build a multi-modal human activity recognition system. In the dataset, we use 5 activities (sitting, sleeping, walking, falling down and jumping), and we record each sample using 3 modalities: infrared array (IRA), WiFi CSI and IMU.

The training set size is 400, and the masked testing set has 63 samples.

### Task 1: Visualize the data (10 pt)
1. Please load one IRA sample of each class from the dataset and visualize it using inferno colormap. Include these images in your report. Specify the sample used.
2. Please select one CSI sample, and visualize the amplitude of the second link and the subcarrier of index 1. Look at the sample below. You need also specify the sample and include the image in your report.

![sample IRA image](output.png)
![sample CSI image](output_wifi.png)

In [1]:
# Task1 visualization — updated to match sample structure
import pickle
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import numpy.core.numeric as numpy_core_numeric

# Compatibility fix for provided pickle files
sys.modules["numpy._core.numeric"] = numpy_core_numeric

DATA_DIR = Path("./data_sources")
OUT_DIR = Path("./output_visualizations")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = {
    "sit":      "20240519162427225649.pickle",
    "walk":     "20240519163436130972.pickle",
    "sleep":    "20240518140039079953.pickle",
    "falldown": "20240521213146575725.pickle",
    "jump":     "20240521213452255777.pickle",
}

def load_sample(fname):
    p = DATA_DIR / fname
    with open(p, "rb") as f:
        return pickle.load(f)

def plot_ira_sample(sample, out_path, title=None, cmap="inferno", vmin=None, vmax=None):
    ira_nodes = sample.get("modality_data", {}).get("IRA", [])
    if not ira_nodes or ira_nodes[0] is None:
        raise ValueError("No IRA frames found in sample")
    frames = np.asarray(ira_nodes[0].get("frames", []))
    if frames.size == 0:
        raise ValueError("Empty IRA frames")
    # frames shape expected (T, H, W) — average (interpolated) over time into one image
    img = frames.mean(axis=0)
    plt.figure(figsize=(4,4))
    im = plt.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    if title:
        plt.title(title)
    plt.axis("off")
    # Colorbar as legend for heatmap
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label("Temperature", rotation=270, labelpad=15)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight", dpi=150)
    plt.close()
    print("Saved IRA image:", out_path)

def plot_wifi_amplitude(sample, out_path, link_idx=1, subcarrier_idx=1, title=None):
    wifi_nodes = sample.get("modality_data", {}).get("wifi", [])
    if not wifi_nodes or wifi_nodes[0] is None:
        raise ValueError("No WiFi frames found in sample")
    frames = np.asarray(wifi_nodes[0].get("frames", []))  # expected (T, links, subcarriers), complex OK
    if frames.size == 0:
        raise ValueError("Empty WiFi frames")
    amp = np.abs(frames)  # amplitude
    if amp.ndim != 3:
        raise ValueError("Unexpected wifi frames shape")
    if link_idx >= amp.shape[1] or subcarrier_idx >= amp.shape[2]:
        raise IndexError("link_idx or subcarrier_idx out of range")
    series = amp[:, link_idx, subcarrier_idx]
    plt.figure(figsize=(6,3))
    plt.plot(series, linewidth=1.2)
    plt.xlabel("Time frame")
    plt.ylabel("Amplitude")
    if title:
        plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print("Saved WiFi amplitude plot:", out_path)

# IRA visualizations
for cls, fname in SAMPLES.items():
    s = load_sample(fname)
    out = OUT_DIR / f"ira_{cls}.png"
    plot_ira_sample(s, out, title=f"IRA - {cls}")

# run WiFi amplitude for an example file
wifi_sample = load_sample(SAMPLES["sit"])
out_wifi = OUT_DIR / "wifi_amp_link2_subcarrier1.png"
plot_wifi_amplitude(wifi_sample, out_wifi, link_idx=1, subcarrier_idx=1,
                    title="CSI Amplitude Link 2, Subcarrier 1")

print("\nAll visualisations completed successfully.")

Saved IRA image: output_visualizations/ira_sit.png
Saved IRA image: output_visualizations/ira_walk.png
Saved IRA image: output_visualizations/ira_sleep.png
Saved IRA image: output_visualizations/ira_falldown.png
Saved IRA image: output_visualizations/ira_jump.png
Saved WiFi amplitude plot: output_visualizations/wifi_amp_link2_subcarrier1.png

All visualisations completed successfully.


### Task 2: Single modality classification (40 pt)
For each modality, build a single modality classifier to recognize the 5 activities. You can choose any model architecture you like, but you need to justify your choice in the report. 

- Marking criteria: 
    - results fully reproducible (10 pt)
    - output accuracy (30 pt)
        - for each modality, highest accuracy (top1) among the class, 10pt
        - top [2 to 20%] accuracy among the class, 9pt
        - top (20% to 50%] accuracy among the class, 8pt
        - top (50% to 70%] accuracy among the class, 6pt
        - rest 30%, runnable code, 5pt

In [1]:
# ============================================================
# Task 2 — Single-modality classification
#
# This cell keeps the original single-modality pipeline but changes the
# validation split so short recordings from the same user/activity session
# stay on the same side of the train/validation boundary.
#
# Each modality is trained for 50 epochs, and the checkpoint with the best
# validation accuracy is kept. The recurrent encoder is bidirectional, while
# hidden size, layer count, and dropout remain unchanged.
# ============================================================

import os
import csv
import random
import pickle
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
import numpy.core.numeric as numpy_core_numeric
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, Subset, DataLoader

# Compatibility fix for provided pickle files
sys.modules["numpy._core.numeric"] = numpy_core_numeric

# Configuration
SEED = 42
DATA_DIR = Path("data_sources")
CSV_FILE = Path("activity_masked.csv")
OUT_DIR = Path("output_visualizations")  # saved models live here
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Determinism
def set_global_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_global_seed(SEED)

# Default shapes used for padding/filling
DEFAULT_SHAPES = {
    "IRA": (1, 24, 32),
    "wifi": (1, 2, 114),
    "imu": (1, 13, 17),
}

# Safe pickle loader
def safe_load_pickle(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        print(f"[load error] {path}: {e}")
        return None

# Dataset class
class sample_dataset(Dataset):
    def __init__(self, dataset_path, dataset_csv=CSV_FILE, read_len=None, validate=True):
        self.dataset_path = Path(dataset_path)
        self.csv_path = Path(dataset_csv)
        self.samples = []
        self.read_csv(read_len)
        if validate:
            self._validate_samples()

    def read_csv(self, read_len=None):
        self.samples = []
        with open(self.csv_path, "r", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            for row in reader:
                if len(row) < 3:
                    continue
                raw_id = (row[2] or "").strip()
                if raw_id in ("", "nan"):
                    continue
                self.samples.append(row)
        if read_len is not None and read_len < len(self.samples):
            self.samples = random.sample(self.samples, read_len)

    def _validate_samples(self):
        kept = []
        for row in self.samples:
            p = Path(self.dataset_path) / row[0]
            if not p.exists():
                print(f"[missing] {row[0]}")
                continue
            data = safe_load_pickle(p)
            if data is None or not isinstance(data, dict) or "modality_data" not in data:
                print(f"[invalid] {row[0]}")
                continue
            kept.append(row)
        print(f"Validated dataset: {len(kept)} / {len(self.samples)} retained")
        self.samples = kept

    def _pad_or_fill_frames(self, frames, default_shape, use_abs=False):
        arr = np.asarray(frames)
        if arr.size == 0:
            return np.zeros(default_shape, dtype=np.float32)
        if use_abs:
            arr = np.abs(arr)
        return arr.astype(np.float32)

    def _normalize_sample(self, data_dict):
        modality_data = data_dict.get("modality_data", {})
        for modality, default_shape in DEFAULT_SHAPES.items():
            node_list = modality_data.get(modality)
            if not node_list:
                modality_data[modality] = [{"frames": np.zeros(default_shape, dtype=np.float32)}]
                continue
            normalized_nodes = []
            for node_item in node_list:
                if node_item is None:
                    normalized_nodes.append({"frames": np.zeros(default_shape, dtype=np.float32)})
                    continue
                normalized_node = dict(node_item)
                normalized_node["frames"] = self._pad_or_fill_frames(
                    node_item.get("frames", []),
                    default_shape,
                    use_abs=(modality == "wifi"),
                )
                normalized_nodes.append(normalized_node)
            modality_data[modality] = normalized_nodes
        data_dict["modality_data"] = modality_data
        return data_dict

    @staticmethod
    def collate_fn(batch):
        modalities = ("IRA", "wifi", "imu")
        default_shapes = DEFAULT_SHAPES
        raw_samples, labels = zip(*batch)
        samples = [dict(s) for s in raw_samples]

        per_modality = {m: [] for m in modalities}
        max_time = {m: 1 for m in modalities}

        for sample in samples:
            modality_data = sample.get("modality_data", {})
            for modality in modalities:
                node_list = modality_data.get(modality, [])
                if not node_list or node_list[0] is None:
                    frames = np.zeros(default_shapes[modality], dtype=np.float32)
                else:
                    frames = np.asarray(node_list[0].get("frames", []))
                    if frames.size == 0:
                        frames = np.zeros(default_shapes[modality], dtype=np.float32)
                    else:
                        if modality == "wifi":
                            frames = np.abs(frames)
                        frames = frames.astype(np.float32)
                per_modality[modality].append(frames)
                max_time[modality] = max(max_time[modality], frames.shape[0])

        batch_modality_data = {}
        for modality in modalities:
            padded = []
            target_t = max_time[modality]
            for frames in per_modality[modality]:
                if frames.shape[0] < target_t:
                    pad_shape = (target_t - frames.shape[0], *frames.shape[1:])
                    pad = np.zeros(pad_shape, dtype=np.float32)
                    frames = np.concatenate([frames, pad], axis=0)
                padded.append(torch.from_numpy(frames))
            batch_modality_data[modality] = torch.stack(padded, dim=0)
        labels_tensor = torch.tensor([int(x) for x in labels], dtype=torch.long)
        user_ids = [s.get("user_id") for s in samples]

        metadata_samples = []
        for sample in samples:
            sample_meta = {}
            for key, value in sample.items():
                if key != "modality_data":
                    sample_meta[key] = value
            modality_meta = {}
            for modality, node_list in sample.get("modality_data", {}).items():
                node_meta_list = []
                for node_item in node_list:
                    if node_item is None:
                        node_meta_list.append(None)
                    else:
                        node_meta = {k: v for k, v in node_item.items() if k != "frames"}
                        node_meta_list.append(node_meta)
                modality_meta[modality] = node_meta_list
            sample_meta.update(modality_meta)
            metadata_samples.append(sample_meta)

        return {
            "metadata": metadata_samples,
            "user_id": user_ids,
            "modality_data": batch_modality_data,
            "label": labels_tensor,
        }

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row = self.samples[idx]
        data_pickle_path = Path(self.dataset_path) / row[0]
        with open(data_pickle_path, "rb") as f:
            data_dict = pickle.load(f)
        data_dict = self._normalize_sample(data_dict)
        data_label = row[2]
        return data_dict, data_label

# Build dataset
set_global_seed(SEED)
dataset_obj = sample_dataset(DATA_DIR, dataset_csv=CSV_FILE, read_len=None, validate=True)
n = len(dataset_obj)
if n == 0:
    raise RuntimeError("No valid labeled samples found in dataset. Fix dataset or validation flag.")

# Split samples by session cluster so nearby windows from the same short
# recording stay together. A session is defined as consecutive samples from
# the same user and activity whose timestamps are no more than 60 seconds
# apart.

def parse_timestamp_from_filename(fname):
    """Filenames look like '20240519162427225649.pickle'. YYYYMMDDHHMMSS + 6-digit micros."""
    stem = fname.split(".")[0]
    if len(stem) < 14:
        return None
    try:
        return datetime.strptime(stem[:14], "%Y%m%d%H%M%S")
    except ValueError:
        return None

def build_session_clusters(samples_rows, user_id_by_filename, gap_seconds=60):
    """Return list of session_ids (same length as samples_rows).
    Two rows share a session iff: same user_id, same activity, and sorted
    timestamps have every consecutive gap <= `gap_seconds`."""
    keyed = []  # (sort_key, original_idx, row)
    for idx, row in enumerate(samples_rows):
        fname, _, act_id = row[0], row[1], row[2]
        ts = parse_timestamp_from_filename(fname)
        uid = user_id_by_filename.get(fname, -1)
        keyed.append(((uid, int(act_id), ts or datetime.min), idx, row))
    # Sort by (user, activity, timestamp) so session-mates are adjacent
    keyed.sort(key=lambda x: x[0])
    session_id_by_idx = [None] * len(samples_rows)
    cur_session = -1
    prev_key = None  # (uid, act, ts)
    for sort_key, orig_idx, row in keyed:
        uid, act, ts = sort_key
        start_new = True
        if prev_key is not None:
            p_uid, p_act, p_ts = prev_key
            if p_uid == uid and p_act == act and p_ts != datetime.min and ts != datetime.min:
                if abs((ts - p_ts).total_seconds()) <= gap_seconds:
                    start_new = False
        if start_new:
            cur_session += 1
        session_id_by_idx[orig_idx] = cur_session
        prev_key = sort_key
    return session_id_by_idx

# Read each sample's user_id so sessions can be clustered by user, activity,
# and timestamp.
user_id_by_filename = {}
for row in dataset_obj.samples:
    data = safe_load_pickle(DATA_DIR / row[0])
    user_id_by_filename[row[0]] = int(data.get("user_id", -1)) if data else -1

session_ids = build_session_clusters(dataset_obj.samples, user_id_by_filename, gap_seconds=60)

# Bucket sessions by (dominant) activity so we can stratify cleanly
session_to_indices = defaultdict(list)
for i, s in enumerate(session_ids):
    session_to_indices[s].append(i)
session_label = {}
for s, idxs in session_to_indices.items():
    labels_here = [int(dataset_obj.samples[i][2]) for i in idxs]
    # sessions are single-activity by construction but we just take the mode
    session_label[s] = max(set(labels_here), key=labels_here.count)

# Stratified 80/20 split over sessions, per activity label
rng_split = random.Random(SEED)
train_indices = []
valid_indices = []
sessions_by_label = defaultdict(list)
for s, lbl in session_label.items():
    sessions_by_label[lbl].append(s)
for lbl, sess_list in sessions_by_label.items():
    rng_split.shuffle(sess_list)
    n_val = max(1, int(round(0.2 * len(sess_list))))  # guarantee at least 1 val session/class
    val_sessions = set(sess_list[:n_val])
    for s in sess_list:
        idxs = session_to_indices[s]
        if s in val_sessions:
            valid_indices.extend(idxs)
        else:
            train_indices.extend(idxs)

# Sanity check: no session should appear in both splits.
assert set(session_ids[i] for i in train_indices).isdisjoint(set(session_ids[i] for i in valid_indices)), \
    "session leakage between train and valid!"

# Worker seeding
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    Subset(dataset_obj, train_indices),
    batch_size=16,
    shuffle=True,
    collate_fn=sample_dataset.collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
valid_loader = DataLoader(
    Subset(dataset_obj, valid_indices),
    batch_size=32,
    shuffle=False,
    collate_fn=sample_dataset.collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

print(f"Dataset: {n} labeled samples; train={len(train_indices)} valid={len(valid_indices)}")


# Bidirectional LSTM encoder with the same hidden size, layer count, and
# dropout as the original setup.
class TemporalLSTMClassifier(nn.Module):
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2, bidirectional=True):
        super().__init__()
        self.input_shape = input_shape
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        flat_dim = int(np.prod(input_shape))
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional,
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Linear(out_dim, 5)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, time_steps = x.shape[0], x.shape[1]
        x = x.view(batch_size, time_steps, -1)
        x = self.proj(x)
        x = self.dropout(x)
        out, (h_n, c_n) = self.lstm(x)
        if self.bidirectional:
            # h_n shape: (num_layers*2, B, hidden); take the last layer fwd+bwd
            last_fwd = h_n[-2]
            last_bwd = h_n[-1]
            last_hidden = torch.cat([last_fwd, last_bwd], dim=1)
        else:
            last_hidden = h_n[-1]
        logits = self.classifier(last_hidden)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_modality_input(batch, modality):
    x = batch["modality_data"][modality].float()
    if modality == "wifi":
        x = x.abs()
    return x

def evaluate_modality(model, loader, modality, device):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            inputs = get_modality_input(batch, modality).to(device)
            labels = batch["label"].to(device) - 1
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

def train_modality(model, train_loader, valid_loader, modality, epochs=15, lr=1e-3, device="cpu"):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_acc = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_total = 0
        for batch in train_loader:
            inputs = get_modality_input(batch, modality).to(device)
            labels = batch["label"].to(device) - 1
            opt.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            opt.step()
            running_loss += loss.item() * labels.size(0)
            running_total += labels.size(0)
        train_loss = running_loss / running_total if running_total > 0 else 0.0
        val_acc = evaluate_modality(model, valid_loader, modality, device)
        print(f"[{modality}] Epoch {ep}/{epochs} train_loss={train_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc

# Input shapes
ira_shape = (24, 32)
wifi_shape = (2, 114)
imu_shape = (13, 17)

# ---- Train and evaluate ----
set_global_seed(SEED)
ira_model = TemporalLSTMClassifier(ira_shape, hidden_size=128, num_layers=2, dropout=0.2, bidirectional=True)
set_global_seed(SEED)
wifi_model = TemporalLSTMClassifier(wifi_shape, hidden_size=128, num_layers=2, dropout=0.2, bidirectional=True)
set_global_seed(SEED)
imu_model = TemporalLSTMClassifier(imu_shape, hidden_size=128, num_layers=2, dropout=0.2, bidirectional=True)

print("Training IRA model...")
ira_model, ira_acc = train_modality(ira_model, train_loader, valid_loader, "IRA", epochs=100, lr=1e-3, device=device)
print("Training WiFi model...")
wifi_model, wifi_acc = train_modality(wifi_model, train_loader, valid_loader, "wifi", epochs=100, lr=1e-3, device=device)
print("Training IMU model...")
imu_model, imu_acc = train_modality(imu_model, train_loader, valid_loader, "imu", epochs=100, lr=1e-3, device=device)

print("\nValidation accuracies -> IRA: {:.4f}, WiFi: {:.4f}, IMU: {:.4f}".format(ira_acc, wifi_acc, imu_acc))

torch.save(ira_model.state_dict(), OUT_DIR / "ira_model.pth")
torch.save(wifi_model.state_dict(), OUT_DIR / "wifi_model.pth")
torch.save(imu_model.state_dict(), OUT_DIR / "imu_model.pth")
print(f"Saved models in {OUT_DIR}")

print("Task 2 finished.")


/tmp/ipykernel_88456/3862552178.py:62: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  return pickle.load(f)


Validated dataset: 400 / 400 retained
Dataset: 400 labeled samples; train=313 valid=87
Training IRA model...


/tmp/ipykernel_88456/3862552178.py:212: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data_dict = pickle.load(f)


[IRA] Epoch 1/100 train_loss=1.5975 val_acc=0.2414
[IRA] Epoch 2/100 train_loss=1.5122 val_acc=0.3333
[IRA] Epoch 3/100 train_loss=1.4468 val_acc=0.4023
[IRA] Epoch 4/100 train_loss=1.4841 val_acc=0.2644
[IRA] Epoch 5/100 train_loss=1.4741 val_acc=0.3793
[IRA] Epoch 6/100 train_loss=1.4619 val_acc=0.3103
[IRA] Epoch 7/100 train_loss=1.4871 val_acc=0.4138
[IRA] Epoch 8/100 train_loss=1.4710 val_acc=0.2989
[IRA] Epoch 9/100 train_loss=1.4447 val_acc=0.3678
[IRA] Epoch 10/100 train_loss=1.4564 val_acc=0.3563
[IRA] Epoch 11/100 train_loss=1.5117 val_acc=0.2069
[IRA] Epoch 12/100 train_loss=1.5324 val_acc=0.3448
[IRA] Epoch 13/100 train_loss=1.4976 val_acc=0.3908
[IRA] Epoch 14/100 train_loss=1.4645 val_acc=0.2989
[IRA] Epoch 15/100 train_loss=1.4309 val_acc=0.3448
[IRA] Epoch 16/100 train_loss=1.4653 val_acc=0.2989
[IRA] Epoch 17/100 train_loss=1.4325 val_acc=0.4138
[IRA] Epoch 18/100 train_loss=1.4540 val_acc=0.3103
[IRA] Epoch 19/100 train_loss=1.4237 val_acc=0.4253
[IRA] Epoch 20/100 tr

Predict and Export CSV

In [4]:
# ============================================================
# Task 2 — Predict masked rows and write per-modality CSVs
#
# Model definitions in this cell must match the training cell so the saved
# checkpoints can be loaded correctly. Labeled rows are copied through
# unchanged, and predictions are written only for masked rows.
# ============================================================
import csv
import pickle
import sys
import numpy as np
import numpy.core.numeric as numpy_core_numeric
import torch
import torch.nn as nn
from pathlib import Path

# Compatibility fix for provided pickle files
sys.modules["numpy._core.numeric"] = numpy_core_numeric

# ========== CONFIGURATION ==========
DATA_DIR = Path("data_sources")
CSV_FILE = Path("activity_masked.csv")
OUT_DIR = Path("output_visualizations")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DEFAULT_SHAPES = {
    "IRA": (1, 24, 32),
    "wifi": (1, 2, 114),
    "imu": (1, 13, 17),
}

def safe_load_pickle(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        print(f"[load error] {path}: {e}")
        return None

def normalize_sample(data_dict):
    modality_data = data_dict.get("modality_data", {})
    for modality, default_shape in DEFAULT_SHAPES.items():
        node_list = modality_data.get(modality)
        if not node_list:
            modality_data[modality] = [{"frames": np.zeros(default_shape, dtype=np.float32)}]
            continue
        normalized_nodes = []
        for node_item in node_list:
            if node_item is None:
                normalized_nodes.append({"frames": np.zeros(default_shape, dtype=np.float32)})
                continue
            normalized_node = dict(node_item)
            frames = np.asarray(node_item.get("frames", []))
            if frames.size == 0:
                frames = np.zeros(default_shape, dtype=np.float32)
            if modality == "wifi":
                frames = np.abs(frames)
            normalized_node["frames"] = frames.astype(np.float32)
            normalized_nodes.append(normalized_node)
        modality_data[modality] = normalized_nodes
    data_dict["modality_data"] = modality_data
    return data_dict

# This classifier matches the architecture used in the training cell.
class TemporalLSTMClassifier(nn.Module):
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2, bidirectional=True):
        super().__init__()
        flat_dim = int(np.prod(input_shape))
        self.bidirectional = bidirectional
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0,
                            bidirectional=bidirectional)
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Linear(out_dim, 5)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch, time = x.shape[0], x.shape[1]
        x = x.view(batch, time, -1)
        x = self.proj(x)
        x = self.dropout(x)
        out, (h_n, _) = self.lstm(x)
        if self.bidirectional:
            last_hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            last_hidden = h_n[-1]
        return self.classifier(last_hidden)

def predict_single_file(model, modality, file_path):
    data = safe_load_pickle(file_path)
    if data is None:
        return None
    data = normalize_sample(data)
    node_list = data["modality_data"].get(modality, [])
    if not node_list or node_list[0] is None:
        arr = np.zeros(DEFAULT_SHAPES[modality], dtype=np.float32)
    else:
        arr = np.asarray(node_list[0].get("frames", []))
        if arr.size == 0:
            arr = np.zeros(DEFAULT_SHAPES[modality], dtype=np.float32)
    t = torch.from_numpy(arr).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        logits = model(t)
        pred = int(logits.argmax(dim=1).item()) + 1
    return pred

def load_model(modality, input_shape, model_path):
    model = TemporalLSTMClassifier(input_shape, hidden_size=128, num_layers=2,
                                   dropout=0.2, bidirectional=True)
    state = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model

# ========== LOAD MODELS ==========
ira_model = load_model("IRA", (24,32), OUT_DIR / "ira_model.pth")
wifi_model = load_model("wifi", (2,114), OUT_DIR / "wifi_model.pth")
imu_model = load_model("imu", (13,17), OUT_DIR / "imu_model.pth")

# ========== READ ORIGINAL CSV ==========
rows = []
with open(CSV_FILE, "r", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)
    rows = list(reader)

def write_pred_csv(model, modality, out_name):
    out_path = Path(out_name)
    out_rows = []
    out_header = ["filename", "activity", "activity_id"]
    masked_count = 0
    for row in rows:
        if len(row) < 3:
            row = row + [""] * (3 - len(row))
        fname = row[0]
        label = (row[2] or "").strip()
        if label == "" or label == "nan":
            p = DATA_DIR / fname
            pred = predict_single_file(model, modality, p)
            masked_count += 1
            if pred is None:
                out_rows.append([fname, "", ""])
            else:
                act_name = {
                    "1": "sitting", "2": "walking", "3": "sleeping",
                    "4": "falldown", "5": "jumping"
                }.get(str(pred), "")
                out_rows.append([fname, act_name, str(pred)])
        else:
            out_rows.append(row)  # preserve ground truth
    with open(out_path, "w", newline="") as wf:
        writer = csv.writer(wf)
        writer.writerow(out_header)
        writer.writerows(out_rows)
    print(f"Wrote {out_path} ({modality})")

write_pred_csv(ira_model, "IRA", "activity_ira.csv")
write_pred_csv(wifi_model, "wifi", "activity_wifi.csv")
write_pred_csv(imu_model, "imu", "activity_imu.csv")


Wrote activity_ira.csv (IRA)
Wrote activity_wifi.csv (wifi)
Wrote activity_imu.csv (imu)


### Task 3: Multi modality classification (10 pt)
Build a multi modality classifier to recognize the 5 activities.

- Marking criteria: 
    - Performance improvement over your group's best single modality classifiers (7 pt)
    - Performance improvement over your group's best single modality classifiers by at least 10% (10 pt)

In [ ]:
# ============================================================
# Task 3 — Multi-modal classification
#
# This cell keeps the original multi-modal workflow while using the same
# grouped train/validation split as the single-modality cell. The encoders
# are bidirectional so the training and prediction cells stay consistent.
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modalities = ["IRA", "wifi", "imu"]
modality_shapes = {"IRA": (24, 32), "wifi": (2, 114), "imu": (13, 17)}

# Bidirectional recurrent encoder used for each modality branch.
class ModalEncoder(nn.Module):
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2, bidirectional=True):
        super().__init__()
        flat_dim = int(input_shape[0] * input_shape[1])
        self.bidirectional = bidirectional
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0,
                            bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = hidden_size * (2 if bidirectional else 1)

    def forward(self, x):
        b, t = x.shape[0], x.shape[1]
        x = x.view(b, t, -1)
        x = self.proj(x)
        x = self.dropout(x)
        _, (h_n, _) = self.lstm(x)
        if self.bidirectional:
            return torch.cat([h_n[-2], h_n[-1]], dim=1)
        return h_n[-1]

class MultiModalLSTMClassifier(nn.Module):
    def __init__(self, modality_shapes, hidden_size=128, num_layers=2, dropout=0.2,
                 num_classes=5, bidirectional=True):
        super().__init__()
        self.modalities = list(modality_shapes.keys())
        self.encoders = nn.ModuleDict({
            m: ModalEncoder(modality_shapes[m], hidden_size, num_layers, dropout, bidirectional)
            for m in self.modalities
        })
        per_enc = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Linear(per_enc * len(self.modalities), hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, batch_modality_dict):
        feats = []
        for m in self.modalities:
            x = batch_modality_dict[m]
            if m == "wifi":
                x = x.abs()
            feats.append(self.encoders[m](x))
        fused = torch.cat(feats, dim=1)
        return self.classifier(fused)

def evaluate_multi(model, loader, device):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            inputs = {m: batch["modality_data"][m].float().to(device) for m in modalities}
            labels = batch["label"].to(device) - 1
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

def train_multi(model, train_loader, valid_loader, epochs=50, lr=1e-3, device=None, verbose=True):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_acc = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_total = 0
        for batch in train_loader:
            inputs = {m: batch["modality_data"][m].float().to(device) for m in modalities}
            labels = batch["label"].to(device) - 1
            opt.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            opt.step()
            running_loss += loss.item() * labels.size(0)
            running_total += labels.size(0)
        train_loss = running_loss / running_total if running_total > 0 else 0.0
        val_acc = evaluate_multi(model, valid_loader, device)
        if verbose:
            print(f"[multi] Epoch {ep}/{epochs} train_loss={train_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc

# ---- Instantiate and train ----
try:
    set_global_seed(SEED)
except Exception:
    pass

multi_model = MultiModalLSTMClassifier(modality_shapes, hidden_size=128, num_layers=2,
                                       dropout=0.2, bidirectional=True)
multi_model, multi_acc = train_multi(multi_model, train_loader, valid_loader, epochs=100, lr=1e-3, device=device)
print(f"Multi-modality validation accuracy: {multi_acc:.4f}")

try:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    pass
torch.save(multi_model.state_dict(), (OUT_DIR / "multi_model.pth"))
print("Saved multi-model to", OUT_DIR / "multi_model.pth")


[multi] Epoch 1/50 train_loss=1.4709 val_acc=0.3793
[multi] Epoch 2/50 train_loss=0.9939 val_acc=0.4828
[multi] Epoch 3/50 train_loss=0.6977 val_acc=0.5287
[multi] Epoch 4/50 train_loss=0.4510 val_acc=0.4138
[multi] Epoch 5/50 train_loss=0.3356 val_acc=0.3908
[multi] Epoch 6/50 train_loss=0.2320 val_acc=0.3218
[multi] Epoch 7/50 train_loss=0.1719 val_acc=0.3563
[multi] Epoch 8/50 train_loss=0.2955 val_acc=0.3908
[multi] Epoch 9/50 train_loss=0.2626 val_acc=0.5632
[multi] Epoch 10/50 train_loss=0.1694 val_acc=0.4598
[multi] Epoch 11/50 train_loss=0.1689 val_acc=0.4598
[multi] Epoch 12/50 train_loss=0.0802 val_acc=0.5632
[multi] Epoch 13/50 train_loss=0.1184 val_acc=0.4713
[multi] Epoch 14/50 train_loss=0.2035 val_acc=0.4483
[multi] Epoch 15/50 train_loss=0.0723 val_acc=0.4943
[multi] Epoch 16/50 train_loss=0.0619 val_acc=0.4483
[multi] Epoch 17/50 train_loss=0.0660 val_acc=0.4598
[multi] Epoch 18/50 train_loss=0.0164 val_acc=0.4828
[multi] Epoch 19/50 train_loss=0.0148 val_acc=0.5517
[m

Predict and Export CSV (MultiModal)

In [6]:
# ============================================================
# Task 3 — Multi-modal prediction + CSV export
#
# The model classes in this cell match the training cell so the saved
# checkpoint can be loaded correctly.
# ============================================================
import csv
import pickle
import sys
import numpy as np
import numpy.core.numeric as numpy_core_numeric
import torch
import torch.nn as nn
from pathlib import Path

# Compatibility fix for provided pickle files
sys.modules["numpy._core.numeric"] = numpy_core_numeric

DATA_DIR = Path("data_sources")
CSV_FILE = Path("activity_masked.csv")
OUT_DIR = Path("output_visualizations")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DEFAULT_SHAPES = {
    "IRA": (1, 24, 32),
    "wifi": (1, 2, 114),
    "imu": (1, 13, 17),
}

def safe_load_pickle(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        print(f"[load error] {path}: {e}")
        return None

def normalize_sample(data_dict):
    modality_data = data_dict.get("modality_data", {})
    for modality, default_shape in DEFAULT_SHAPES.items():
        node_list = modality_data.get(modality)
        if not node_list:
            modality_data[modality] = [{"frames": np.zeros(default_shape, dtype=np.float32)}]
            continue
        normalized_nodes = []
        for node_item in node_list:
            if node_item is None:
                normalized_nodes.append({"frames": np.zeros(default_shape, dtype=np.float32)})
                continue
            normalized_node = dict(node_item)
            frames = np.asarray(node_item.get("frames", []))
            if frames.size == 0:
                frames = np.zeros(default_shape, dtype=np.float32)
            if modality == "wifi":
                frames = np.abs(frames)
            normalized_node["frames"] = frames.astype(np.float32)
            normalized_nodes.append(normalized_node)
        modality_data[modality] = normalized_nodes
    data_dict["modality_data"] = modality_data
    return data_dict

# This encoder matches the architecture used in the training cell.
class ModalEncoder(nn.Module):
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2, bidirectional=True):
        super().__init__()
        flat_dim = int(input_shape[0] * input_shape[1])
        self.bidirectional = bidirectional
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0,
                            bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, t = x.shape[0], x.shape[1]
        x = x.view(b, t, -1)
        x = self.proj(x)
        x = self.dropout(x)
        _, (h_n, _) = self.lstm(x)
        if self.bidirectional:
            return torch.cat([h_n[-2], h_n[-1]], dim=1)
        return h_n[-1]

class MultiModalLSTMClassifier(nn.Module):
    def __init__(self, modality_shapes, hidden_size=128, num_layers=2, dropout=0.2,
                 num_classes=5, bidirectional=True):
        super().__init__()
        self.modalities = list(modality_shapes.keys())
        self.encoders = nn.ModuleDict({
            m: ModalEncoder(modality_shapes[m], hidden_size, num_layers, dropout, bidirectional)
            for m in self.modalities
        })
        per_enc = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Linear(per_enc * len(self.modalities), hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, batch_modality_dict):
        feats = []
        for m in self.modalities:
            x = batch_modality_dict[m]
            if m == "wifi":
                x = x.abs()
            feats.append(self.encoders[m](x))
        fused = torch.cat(feats, dim=1)
        return self.classifier(fused)

# Load saved multi-model
modality_shapes = {"IRA": (24, 32), "wifi": (2, 114), "imu": (13, 17)}
multi_path = OUT_DIR / "multi_model.pth"
if not multi_path.exists():
    raise FileNotFoundError(f"Multi-model checkpoint not found: {multi_path}")

multi_model = MultiModalLSTMClassifier(modality_shapes, hidden_size=128, num_layers=2,
                                       dropout=0.2, bidirectional=True)
state = torch.load(multi_path, map_location=DEVICE)
multi_model.load_state_dict(state)
multi_model.to(DEVICE)
multi_model.eval()

def predict_single_file_multi(model, file_path):
    data = safe_load_pickle(file_path)
    if data is None:
        return None
    data = normalize_sample(data)
    inputs = {}
    for m in ["IRA", "wifi", "imu"]:
        node_list = data["modality_data"].get(m, [])
        if not node_list or node_list[0] is None:
            arr = np.zeros(DEFAULT_SHAPES[m], dtype=np.float32)
        else:
            arr = np.asarray(node_list[0].get("frames", []))
            if arr.size == 0:
                arr = np.zeros(DEFAULT_SHAPES[m], dtype=np.float32)
        if m == "wifi":
            arr = np.abs(arr)
        t = torch.from_numpy(arr).unsqueeze(0).float().to(DEVICE)
        inputs[m] = t
    with torch.no_grad():
        logits = model(inputs)
        pred = int(logits.argmax(dim=1).item()) + 1
    return pred

rows = []
with open(CSV_FILE, "r", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)
    rows = list(reader)

def write_pred_csv_multi(model, out_name="activity_multi.csv"):
    out_path = Path(out_name)
    out_rows = []
    out_header = ["filename", "activity", "activity_id"]
    masked_count = 0
    for row in rows:
        if len(row) < 3:
            row = row + [""] * (3 - len(row))
        fname = row[0]
        label = (row[2] or "").strip()
        if label == "" or label == "nan":
            p = DATA_DIR / fname
            pred = predict_single_file_multi(model, p)
            masked_count += 1
            if pred is None:
                out_rows.append([fname, "", ""])
            else:
                act_name = {
                    "1": "sitting", "2": "walking", "3": "sleeping",
                    "4": "falldown", "5": "jumping"
                }.get(str(pred), "")
                out_rows.append([fname, act_name, str(pred)])
        else:
            out_rows.append(row)
    with open(out_path, "w", newline="") as wf:
        writer = csv.writer(wf)
        writer.writerow(out_header)
        writer.writerows(out_rows)
    print(f"Wrote {out_path} (multi)")

write_pred_csv_multi(multi_model, "activity_multi.csv")


Wrote activity_multi.csv (multi)


### Task 4: Report (40 pt)
Write a report of around 4 pages about your findings, runtime and results. Use the same latex template as in the individual project. Put down the group members' name, student ID and email in the report.

In the report, include all your processing steps (include the AI workflows), as well as the GenAI declaration. Introduce the contribution of each group member in the report. 

- Marking criteria: 
    - Clarity of the report (10 pt)
    - Justification of the model architecture choices (10 pt)
    - Runtime, results, and analysis (10 pt)
    - Processing pipelines, AI workflows, etc (10 pt)

## Submission requirements:
__Pack the following materials into a zip file:__
2. all your code and trained checkpoints
3. the predicted csv with the same format as the training csv provided. Specifically, we use such a mapping for the activity labels:
    ```
        - 1: sitting
        - 2: walking
        - 3: sleeping
        - 4: falling down
        - 5: jumping
    ```
    The csv we provide contains 3 columns, filename,activity,activity_id. For the masked test set, the last two columns are blank. You should fill in the blank fields in the csv provided with your predictions, and submit the csv file. __You need to submit 4 csv files, with name activity_wifi.csv, activity_imu.csv, activity_ira.csv, activity_multi.csv, to represent the corresponding modality used.__
4. the report (pdf file) generated from the latex template provided. You can modify the body.tex file to write your report. Include all your group members' name, student ID and email in the report. Also include the visualized images in the report.
5. a short README.md file to the code structure and important steps to reproduce your results. You should make sure that your code is compatible with the provided data files and metadata. Include the library requirements if you use any additional libraries, and indicate this in the README.md file.

Each group only need to submit one copy. 